# Generate Disease Ontology Multilabel Class Vectors

The idea here is to generate multilabel class binary vectors with which to later fine tune scGPT models. 

In [ ]:
import importlib
import method_utils  # your module

importlib.reload(method_utils)

<module 'method_utils' from '/aloy/home/ddalton/projects/scGPT_playground/notebooks/method_utils.py'>

In [2]:
"""Generate DO Multilabel Class Vectors

Structure:
    1. Imports, Variables, Functions
    2. Load Data
    3. Generate Multilabel Vectors
"""


# 1. Imports, Variables, Functions
# imports
import method_utils as mu
import sys, os, numpy as np, pandas as pd, pickle
import networkx as nx, obonet, scanpy as sc

# variables
data_dir = "/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-05-07-01"



# functions
def get_node_info(node, do_g, do_sanchez_ic):
    return {
        "id": node,
        "name": do_g.nodes[node].get("name", "Unknown"),
        "sanchez_ic": do_sanchez_ic.get(node, None),
    }


# 2. Load Data
# load the DO graph
do_g = mu.load_do_graph()

# get sanchez information content
doid_2_ic = mu.get_sanchez_ic(do_g)
print(f"Max IC: {max(doid_2_ic.values())}")
print(f"Min IC: {min(doid_2_ic.values())}")
ic_thr = np.percentile(list(doid_2_ic.values()), 0.5)
print(f"5th percentile IC threshold: {ic_thr}")
print(f"Nº of nodes with IC < {ic_thr}: {sum(ic < ic_thr for ic in doid_2_ic.values())}")
for node, ic in doid_2_ic.items():
    if (ic < ic_thr) and (node != "DOID:4"):
        print(f"Node {node} - {do_g.nodes[node].get('name', 'Unknown')} has IC {ic}, below threshold {ic_thr}")

# mapping from UMLS to DOID
umls_2_doid = mu.get_umls_2_doid_mapping(do_g)

# load the AnnData object
adata = sc.read_h5ad(os.path.join(data_dir, "data.h5ad"))


Number of DO leaves: 9018
Max IC: 11.746146071780966
Min IC: 0.6931471805599453
5th percentile IC threshold: 5.505018816009248
Nº of nodes with IC < 5.505018816009248: 57
Node DOID:0014667 - disease of metabolism has IC 3.688029757921543, below threshold 5.505018816009248
Node DOID:0050117 - disease by infectious agent has IC 4.153611861909401, below threshold 5.505018816009248
Node DOID:0050155 - sensory system disease has IC 3.9434462789442732, below threshold 5.505018816009248
Node DOID:0050177 - monogenic disease has IC 2.2551104786525693, below threshold 5.505018816009248
Node DOID:0050686 - organ system cancer has IC 3.477311680820003, below threshold 5.505018816009248
Node DOID:0050687 - cell type cancer has IC 4.185649026873923, below threshold 5.505018816009248
Node DOID:0050735 - X-linked monogenic disease has IC 4.986426871626234, below threshold 5.505018816009248
Node DOID:0050736 - autosomal dominant disease has IC 3.6596332376139573, below threshold 5.505018816009248
Node

In [41]:

# 3. Generate Multilabel Vectors
# A. Select level 1 nodes
# get level 1 nodes
lvl1_nodes = mu.get_lvl1_nodes(do_g)

# Generate multilabel vectors for level 1 nodes
Y_multilabel_lvl1, lvl1_nodes = mu.generate_multilabel_vectors(adata, do_g, lvl1_nodes)
print(f"Generated multilabel vectors for level 1 nodes with shape {Y_multilabel_lvl1.shape}")

# Clean multilabel vectors by removing nodes with no samples
Y_multilabel_lvl1, lvl1_nodes = mu.clean_multilabel_vectors(Y_multilabel_lvl1, lvl1_nodes)
print(f"Cleaned multilabel vectors for top 50 nodes with shape {Y_multilabel_lvl1.shape}")

# Check the multilabel vector for level 1 nodes
mu.check_multilabel_vector(Y_multilabel_lvl1, lvl1_nodes, do_g)

# get doids and class names
Y_multilabel_lvl1_doid, Y_multilabel_lvl1_name = mu.get_multilabel_data(Y_multilabel_lvl1, lvl1_nodes, do_g)

# # save the multilabel vectors for level 1 nodes
# _output_dir = os.path.join(os.path.dirname(data_dir), os.path.split(data_dir)[-1] + ".lvl1_label")
# if not os.path.exists(_output_dir):
#     os.makedirs(_output_dir)

# # save addata object
# adata_2 = adata.copy()  # Create a copy of the original AnnData object
# adata_2.write(os.path.join(_output_dir, "data.h5ad"))

# save the multilabel vectors for level 1 nodes
Y_data_lvl1 = {
    "loaded_Y_data_lvl1": Y_multilabel_lvl1,
    "Y_multilabel_doid": Y_multilabel_lvl1_doid,
    "Y_multilabel_name": Y_multilabel_lvl1_name,
    "nodes": lvl1_nodes,
}


# pickle.dump(Y_data_lvl1, open(os.path.join(_output_dir, "Y_multilabel.pkl"), "wb"))

# # load the multilabel vectors for level 1 nodes
# loaded_Y_data_lvl1 = pickle.load(open(os.path.join(_output_dir, "Y_multilabel.pkl"), "rb"))


Root node: DOID:4 - disease
Nº Level 1 nodes: 8
Generated multilabel vectors for level 1 nodes with shape (20932, 8)
Cleaned multilabel vectors for top 50 nodes with shape (20932, 6)
250 samples	Node DOID:0014667 - disease of metabolism
1739 samples	Node DOID:0050117 - disease by infectious agent
2784 samples	Node DOID:14566 - disease of cellular proliferation
431 samples	Node DOID:150 - disease of mental health
866 samples	Node DOID:630 - genetic disease
16156 samples	Node DOID:7 - disease of anatomical entity
Nº of samples with only one label: 19638
Nº of samples with +1 labels: 1294
Max nº of labels per sample: 2


In [3]:
def get_node_lvl(do_g: nx.Graph, node: str) -> int:
    """
    Get the level of a node in the Directed Ontology Graph (DO Graph).
    """    
    # count shortest path from root to node
    root_nodes = [n for n in do_g.nodes if do_g.in_degree(n) == 0]
    assert len(root_nodes) == 1, "Multiple root nodes found!"
    root = root_nodes[0]

    return nx.shortest_path_length(do_g, source=root, target=node)




def get_lvl1_nodes(do_g: nx.Graph) -> list:
    # get root node
    root_nodes = [n for n in do_g.nodes if do_g.in_degree(n) == 0]
    assert len(root_nodes) == 1, "Multiple root nodes found!"
    root = root_nodes[0]
    print(f"Root node: {root} - {do_g.nodes[root].get('name', 'Unknown')}")

    # get level 1 
    lvl1_nodes = list(do_g.successors(root))  # First level children
    print(f"Nº Level 1 nodes: {len(lvl1_nodes)}")

    return lvl1_nodes

In [10]:
grouped_nodes = adata.obs["do_id"].unique()

# which level are these grouped nodes from?
g = [get_node_lvl(do_g, node) for node in grouped_nodes]
for node, level in zip(grouped_nodes, g):
    print(f"Node {node} is at level {level} - {do_g.nodes[node].get('name', 'Unknown')}")

Node DOID:0080000 is at level 3 - muscular disease
Node DOID:0050117 is at level 1 - disease by infectious agent
Node DOID:65 is at level 3 - connective tissue disease
Node DOID:1287 is at level 2 - cardiovascular system disease
Node DOID:15 is at level 2 - reproductive system disease
Node DOID:16 is at level 2 - integumentary system disease
Node DOID:0050155 is at level 3 - sensory system disease
Node DOID:0060083 is at level 3 - immune system cancer
Node DOID:1579 is at level 2 - respiratory system disease
Node DOID:74 is at level 2 - hematopoietic system disease
Node DOID:77 is at level 2 - gastrointestinal system disease
Node DOID:0014667 is at level 1 - disease of metabolism
Node DOID:0050735 is at level 3 - X-linked monogenic disease
Node DOID:0050687 is at level 3 - cell type cancer
Node DOID:18 is at level 2 - urinary system disease
Node DOID:936 is at level 4 - brain disease
Node DOID:0080014 is at level 2 - chromosomal disease
Node DOID:612 is at level 3 - primary immunodefic

In [21]:
# get all level 1, 2 and 3 nodes which are NOT leaf nodes
lvl1_nodes = get_lvl1_nodes(do_g)
lvl2_nodes = [n for lvl1_node in lvl1_nodes for n in do_g.successors(lvl1_node) if do_g.out_degree(n) > 0]
lvl3_nodes = [n for lvl2_node in lvl2_nodes for n in do_g.successors(lvl2_node) if do_g.out_degree(n) > 0]

print(f"Nº Level 1 nodes: {len(lvl1_nodes)}")
print(f"Nº Level 2 nodes: {len(lvl2_nodes)}")
print(f"Nº Level 3 nodes: {len(lvl3_nodes)}")


# which nodes have no leaf terms under and only have generic terms?

def is_generic_node(do_g: nx.Graph,node: str) -> bool:
    """
    Check if a node is generic, meaning it has no leaf nodes under it.
    """
    # get all successors of the node
    successors = list(do_g.successors(node))
    
    # check if any of the successors are leaf nodes
    for succ in successors:
        if do_g.out_degree(succ) == 0:  # If it has no children, it's a leaf node
            return False
    
    return True  # If all successors are non-leaf, it's a generic node

print(f"Nº of generic nodes at level 1: {sum(is_generic_node(do_g, node) for node in lvl1_nodes)}")
print(f"Nº of generic nodes at level 2: {sum(is_generic_node(do_g, node) for node in lvl2_nodes)}")
print(f"Nº of generic nodes at level 3: {sum(is_generic_node(do_g, node) for node in lvl3_nodes)}")

Root node: DOID:4 - disease
Nº Level 1 nodes: 8
Nº Level 1 nodes: 8
Nº Level 2 nodes: 146
Nº Level 3 nodes: 253
Nº of generic nodes at level 1: 5
Nº of generic nodes at level 2: 14
Nº of generic nodes at level 3: 25


In [ ]:
adata_2 = adata.copy()  # Create a copy of the original AnnData object
adata_2.obs = mu.add_multilabel_to_adata(adata_2, Y_multilabel_lvl1, Y_multilabel_lvl1_doid, Y_multilabel_lvl1_name)

In [49]:
y_data = mu.get_multilabel_dict_from_adata(adata_2)

In [22]:
np.array(y_data["Y_multilabel"])

NameError: name 'y_data' is not defined

In [ ]:
# B. Select lowest 50 IC nodes

bot_50_nodes = mu.get_n_lowest_ic_nodes(doid_2_ic, n=50)

# # check if all children of a node are in the list
# for node in bot_50_nodes:
#     children = list(do_g.successors(node))

#     if all(child in bot_50_nodes for child in children):
#         print(f"Node {node} has all children in the list: {children} - {[do_g.nodes[child].get('name', 'Unknown') for child in children]}")
    
#         # remove the node 
#         bot_50_nodes.remove(node)
#         print(f"Removed node {node} - {do_g.nodes[node].get('name', 'Unknown')} from the list, as all its children are in the list.")

assert len(bot_50_nodes) == 50, "Expected 50 nodes, got {}".format(len(bot_50_nodes))

# get IC for all of these nodes
print(f"Max IC: {max(doid_2_ic[n] for n in bot_50_nodes)}")

# check if it contains all level 1 nodes
print(f"Contains all level 1 nodes: {all(n in bot_50_nodes for n in lvl1_nodes)}")

# Generate multilabel vectors for bot 50 nodes
Y_multilabel_50, bot_50_nodes = mu.generate_multilabel_vectors(adata, do_g, bot_50_nodes)
print(f"Generated multilabel vectors for top 50 nodes with shape {Y_multilabel_50.shape}")

Y_multilabel_50, bot_50_nodes = mu.clean_multilabel_vectors(Y_multilabel_50, bot_50_nodes)
print(f"Cleaned multilabel vectors for top 50 nodes with shape {Y_multilabel_50.shape}")

mu.check_multilabel_vector(Y_multilabel_50, bot_50_nodes, do_g)


Y_multilabel_50_doid, Y_multilabel_50_name = mu.get_multilabel_data(Y_multilabel_50, bot_50_nodes, do_g)

# Add multilabel vectors to AnnData object
adata_2 = adata.copy()  # Create a copy of the original AnnData object
adata_2.obs = mu.add_multilabel_to_adata(adata, Y_multilabel_50, Y_multilabel_50_doid, Y_multilabel_50_name)

Max IC: 5.364668521123742
Contains all level 1 nodes: True
Generated multilabel vectors for top 50 nodes with shape (20932, 50)
Cleaned multilabel vectors for top 50 nodes with shape (20932, 27)
250 samples	Node DOID:0014667 - disease of metabolism
1739 samples	Node DOID:0050117 - disease by infectious agent
284 samples	Node DOID:0050155 - sensory system disease
172 samples	Node DOID:0050177 - monogenic disease
1294 samples	Node DOID:0050686 - organ system cancer
1490 samples	Node DOID:0050687 - cell type cancer
172 samples	Node DOID:0050735 - X-linked monogenic disease
560 samples	Node DOID:0080000 - muscular disease
890 samples	Node DOID:1287 - cardiovascular system disease
2784 samples	Node DOID:14566 - disease of cellular proliferation
412 samples	Node DOID:15 - reproductive system disease
431 samples	Node DOID:150 - disease of mental health
747 samples	Node DOID:1579 - respiratory system disease
2000 samples	Node DOID:16 - integumentary system disease
2784 samples	Node DOID:162 - 

In [ ]:
# generate small benchmark set

_class_nodes = adata.obs["do_id"].unique()
print(f"Nº of class nodes: {len(class_nodes)}")

# Generate multilabel vectors for level 1 nodes
Y_multilabel, _class_nodes = mu.generate_multilabel_vectors(adata, do_g, _class_nodes)
print(f"Generated multilabel vectors for level 1 nodes with shape {Y_multilabel.shape}")

# Clean multilabel vectors by removing nodes with no samples
Y_multilabel, _class_nodes = mu.clean_multilabel_vectors(Y_multilabel, _class_nodes)
print(f"Cleaned multilabel vectors for top 50 nodes with shape {Y_multilabel.shape}")

# Check the multilabel vector for level 1 nodes
mu.check_multilabel_vector(Y_multilabel, _class_nodes, do_g)

Nº of class nodes: 19
Generated multilabel vectors for level 1 nodes with shape (20932, 19)
Cleaned multilabel vectors for top 50 nodes with shape (20932, 19)
250 samples	Node DOID:0014667 - disease of metabolism
1739 samples	Node DOID:0050117 - disease by infectious agent
284 samples	Node DOID:0050155 - sensory system disease
1490 samples	Node DOID:0050687 - cell type cancer
172 samples	Node DOID:0050735 - X-linked monogenic disease
1294 samples	Node DOID:0060083 - immune system cancer
560 samples	Node DOID:0080000 - muscular disease
694 samples	Node DOID:0080014 - chromosomal disease
890 samples	Node DOID:1287 - cardiovascular system disease
412 samples	Node DOID:15 - reproductive system disease
431 samples	Node DOID:1561 - cognitive disorder
747 samples	Node DOID:1579 - respiratory system disease
2000 samples	Node DOID:16 - integumentary system disease
538 samples	Node DOID:18 - urinary system disease
283 samples	Node DOID:612 - primary immunodeficiency disease
5503 samples	Node DOI

: 